In [11]:
# import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [12]:
# import database / data

In [13]:
import pandas as pd
import sqlite3

# 1. Connect to the SQLite database
conn = sqlite3.connect('customer_churn_data_raw.db')

# 2. Read the Excel file and load all its sheets into the SQLite database
excel_file = 'customer_churn_data_raw.xlsx'
sheet_names = pd.ExcelFile(excel_file).sheet_names

for sheet in sheet_names:
    # Read each sheet into a temporary dataframe
    temp_df = pd.read_excel(excel_file, sheet_name=sheet)
    # Save it as a table in the database
    temp_df.to_sql(sheet, conn, if_exists='replace', index=False)
    print(f"Loaded sheet '{sheet}' into database table '{sheet}'")

print("-" * 30)

# 3. Now run your original query
sql_query = """
        select name
        from sqlite_master
        where type = 'table'
"""
tables = pd.read_sql(sql_query, conn)

# 4. Create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"created dataframe: df_{table_name}")

conn.close()


Loaded sheet 'db_customer' into database table 'db_customer'
Loaded sheet 'db_subscription' into database table 'db_subscription'
Loaded sheet 'db_support' into database table 'db_support'
------------------------------
created dataframe: df_db_customer
created dataframe: df_db_subscription
created dataframe: df_db_support


In [15]:
# print table names and column names

conn = sqlite3.connect('customer_churn_data_raw.db')

for table_name in tables['name']:
    print(f"\n Table Name: {table_name}")
    # get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("columns:")
    print(columns['name'].tolist())

# close connection
conn.close()


 Table Name: db_customer
columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

 Table Name: db_subscription
columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

 Table Name: db_support
columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']
